# Bulk CSV → 100m WGS84 Raster Pipeline

This notebook processes Google Open Buildings CSV.gz files into 100m resolution WGS84 rasters with area-accurate building footprint coverage (m²).

**Key Features:**
- Adaptive splitting for 300+ CSV files
- Parallel conversion (CSV.gz → GPKG)
- 100m UTM grid with exact area computation per cell
- PostGIS-backed with resume support
- Multi-platform (Windows/Linux with automatic executor selection)

## Prerequisites

### System Dependencies
Install GDAL, PostgreSQL, and Python packages:

**Ubuntu/Debian:**
```bash
apt-get install -y gdal-bin postgresql postgresql-contrib python3-pip
```

**Conda (Recommended):**
```bash
conda install -y -c conda-forge gdal postgresql rasterio fiona psycopg2 numpy psutil pyproj
```

### Python Packages
```bash
pip install numpy rasterio fiona psycopg2-binary psutil pyproj
```

### PostgreSQL Setup
```bash
createdb buildings
psql -d buildings -c "CREATE EXTENSION postgis;"
```

## Input & Output Specifications

### Input Data

**Source Format:** Google Open Buildings CSV.gz files
- **Location:** `/data/csv` (or `CSV_DIR` environment variable)
- **File format:** Gzipped CSV with header row
- **Expected columns:**
  - `geometry` (WKT polygon in EPSG:4326)
  - Other columns: preserved but optional for rasterization

**Example input structure:**
```
geometry,confidence,area_in_meters
"POLYGON ((61.0 29.5, 61.1 29.5, 61.1 29.6, 61.0 29.6, 61.0 29.5))",0.8,150
```

**Input requirements:**
- Valid WGS84 coordinates (-180 to 180 longitude, -90 to 90 latitude)
- Polygonal geometries (points/lines will be converted or skipped)
- UTF-8 encoding with gzip compression
- Size: Can be 300+ files, from 1 MB to 10+ GB per file

---

### Output Data

**Output Format:** GeoTIFF rasters at 100m resolution, WGS84 projection (EPSG:4326)

**Primary Output:** Individual raster per source
- **Location:** `{RASTER_DIR}/tifs/` (default: `/data/rasters/tifs/`)
- **Naming convention:** `{source_name}_100m.tif`
- **Data type:** Float32
- **Pixel values:** Building footprint area in m² per 100m × 100m cell
  - Range: 0 to 10,000 m² (clamped to avoid overflow)
  - No data value: -9999 (represents no buildings in cell)
- **File size:** Varies with geographic extent; typically 10-500 MB per source

**Secondary Output:** Mosaic GeoTIFF combining all sources
- **Location:** `{RASTER_DIR}/mosaic_100m_{TIMESTAMP}.tif`
- **Coverage:** Union of all input extents, globally aligned to (-180, -90) origin
- **Resolution:** 100m × 100m cells (≈1.1 km² per pixel at equator)
- **Compression:** LZW, tiled storage for efficient access
- **Metadata:** Coordinate system (EPSG:4326), NoData value (-9999), timestamp

**Tracking & Logs:**
- **Manifest:** `{LOG_DIR}/raster_manifest.json` - Resume tracking (base name → TIF path, file size, timestamp)
- **Pipeline log:** `{LOG_DIR}/bulk_pipeline_{TIMESTAMP}.log` - Processing steps and timings
- **Intermediate tables (temporary):** PostGIS tables (`src_{base}`, `grid_{base}`, `area_{base}`) - auto-dropped after rasterization

**Intermediate Files (can be cleaned up):**
- **GPKG files:** `{GPKG_DIR}/*.gpkg` - OGR intermediate format (can delete after successful rasterization)
- **Split CSV chunks:** `{SPLIT_ROOT}/*.csv.gz` - For large files split during processing (can delete after conversion)
- **VRT files:** `{RASTER_DIR}/tifs/mosaic.vrt` - Virtual mosaic (temporary)

---

### Data Alignment & Projection

**Coordinate System:** EPSG:4326 (WGS84)
- Longitude: -180° to +180°
- Latitude: -90° to +90°

**Grid Alignment:**
- All rasters aligned to global 100m grid with origin at (-180°, -90°)
- Cell boundaries computed in UTM (per-region) then back-transformed to WGS84
- Enables seamless mosaicking and tiling across continents
- Pixel size: ~111.3 m (longitude) × ~111.3 m (latitude) at equator

**Area Computation:**
1. Internal processing in UTM projection (meter-based coordinates)
2. Building footprints extracted from source polygons
3. Exact area intersection per 100m grid cell computed using `ST_Intersection`
4. Clamped to 10,000 m² max (100×100m cell theoretical max)
5. Final raster back-transformed to WGS84 for output

In [ ]:
# 1. Core Imports & Configuration
import os
import sys
import json
import time
import math
import gzip
import subprocess
from pathlib import Path
from typing import List, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configuration
class Config:
    # Directories
    CSV_DIR = Path(os.environ.get('CSV_DIR', '/data/csv'))
    SPLIT_ROOT = Path(os.environ.get('SPLIT_ROOT', '/data/splits'))
    GPKG_DIR = Path(os.environ.get('GPKG_DIR', '/data/gpkg'))
    RASTER_DIR = Path(os.environ.get('RASTER_DIR', '/data/rasters'))
    LOG_DIR = Path(os.environ.get('LOG_DIR', '/data/logs'))
    
    # Database
    DB_NAME = os.environ.get('BUILDINGS_DB_NAME', 'buildings')
    DB_USER = os.environ.get('BUILDINGS_DB_USER', 'postgres')
    DB_HOST = os.environ.get('BUILDINGS_DB_HOST', 'localhost')
    DB_PORT = os.environ.get('BUILDINGS_DB_PORT', '5432')
    DB_PASSWORD = os.environ.get('BUILDINGS_DB_PASSWORD')
    
    # Performance
    CHUNK_ROWS = int(os.environ.get('CHUNK_ROWS', '1000000'))
    WORKERS = max(1, (os.cpu_count() or 8) // 2)
    
    # Grid alignment
    ALIGN_ORIGIN = (-180.0, -90.0)
    CELL_SIZE_M = 100.0
    AREA_CLAMP = 10000.0  # m²

# Create directories
for d in [Config.SPLIT_ROOT, Config.GPKG_DIR, Config.RASTER_DIR / 'tifs', Config.LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime('%Y%m%d_%H%M%S')
print(f'✓ Configuration loaded (RUN_ID={RUN_ID})')
print(f'  Workers: {Config.WORKERS}')
print(f'  Chunk size: {Config.CHUNK_ROWS} rows')

In [ ]:
# 2. Utility Functions
import shutil

def check_tools():
    """Verify required GDAL and PostgreSQL tools."""
    required = ['ogr2ogr', 'gdal_rasterize', 'gdalbuildvrt', 'psql']
    missing = [t for t in required if not shutil.which(t)]
    if missing:
        raise EnvironmentError(f'Missing tools: {missing}')
    print('✓ All required tools found')

def run_cmd(cmd, timeout=300):
    """Run shell command and return (returncode, stdout, stderr)."""
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return result.returncode, result.stdout, result.stderr
    except subprocess.TimeoutExpired:
        return 1, '', f'Command timed out after {timeout}s'
    except Exception as e:
        return 1, '', str(e)

def safe_name(text):
    """Sanitize base names for table/filename use."""
    import re
    text = (text or '').strip().lower()
    text = re.sub(r'[^0-9a-z_]+', '_', text)
    text = text.strip('_')
    return text[:80] if text else 'tile'

check_tools()

In [ ]:
# 3. CSV Splitting & Manifest
def split_csv_gz(src, out_dir, rows_per_chunk):
    """Split gzipped CSV into chunks."""
    out_dir.mkdir(parents=True, exist_ok=True)
    base = src.stem.replace('.csv', '')
    chunks = []
    idx = 0
    written = 0
    fout = None
    
    with gzip.open(src, 'rt', encoding='utf-8', errors='ignore') as fin:
        header = fin.readline()
        if not header:
            return []
        
        for line in fin:
            if written == 0:
                idx += 1
                fout = gzip.open(out_dir / f'{base}.part{idx:04d}.csv.gz', 'wt', encoding='utf-8')
                fout.write(header)
            
            fout.write(line)
            written += 1
            
            if written >= rows_per_chunk:
                fout.close()
                fout = None
                written = 0
        
        if fout:
            fout.close()
    
    for i in range(1, idx + 1):
        chunks.append(out_dir / f'{base}.part{i:04d}.csv.gz')
    
    return chunks

# Find all CSV files
all_csv = sorted(Config.CSV_DIR.glob('*.csv.gz'))
print(f'Found {len(all_csv)} CSV files')

# Partition by size
LARGE_SIZE = 2000 * 1024 * 1024  # 2 GB
large_files = [f for f in all_csv if f.stat().st_size >= LARGE_SIZE]
small_files = [f for f in all_csv if f.stat().st_size < LARGE_SIZE]
print(f'  Large (>2GB): {len(large_files)}')
print(f'  Small (<2GB): {len(small_files)}')

In [ ]:
# 4. Parallel CSV-to-GPKG Conversion
def convert_csv_to_gpkg(csv_path):
    """Convert CSV.gz to GPKG using ogr2ogr."""
    gpkg = Config.GPKG_DIR / csv_path.stem.replace('.csv', '.gpkg')
    
    # Skip if already exists
    if gpkg.exists() and gpkg.stat().st_size > 0:
        return True
    
    cmd = (
        f'ogr2ogr -f GPKG "{gpkg}" '
        f'-oo AUTODETECT_TYPE=NO '
        f'-oo GEOM_POSSIBLE_NAMES=geometry '
        f'-nlt PROMOTE_TO_MULTI '
        f'-nln buildings '
        f'/vsigzip/"{csv_path}"'
    )
    
    rc, _, err = run_cmd(cmd)
    
    if rc == 0 and gpkg.exists() and gpkg.stat().st_size > 0:
        return True
    
    print(f'⚠ Conversion failed for {csv_path.name}: {err[-200:]}')
    return False

# Convert all CSVs
convert_targets = small_files.copy()
for large_file in large_files:
    split_dir = Config.SPLIT_ROOT / large_file.stem
    chunks = split_csv_gz(large_file, split_dir, Config.CHUNK_ROWS)
    convert_targets.extend(chunks)
    print(f'Split {large_file.name} into {len(chunks)} chunks')

print(f'\nConverting {len(convert_targets)} CSV files to GPKG...')
success = 0
with ThreadPoolExecutor(max_workers=Config.WORKERS) as ex:
    for ok in ex.map(convert_csv_to_gpkg, convert_targets):
        if ok:
            success += 1
        # Progress
        if (success + 1) % max(1, len(convert_targets) // 10) == 0:
            print(f'  {success}/{len(convert_targets)} converted')

print(f'✓ Conversion complete: {success}/{len(convert_targets)} succeeded')

In [ ]:
# 5. PostgreSQL Setup
try:
    import psycopg2
except ImportError:
    print('Installing psycopg2...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'psycopg2-binary'])
    import psycopg2

def get_db_connection():
    """Get PostgreSQL connection."""
    kwargs = {
        'dbname': Config.DB_NAME,
        'user': Config.DB_USER,
        'host': Config.DB_HOST,
        'port': Config.DB_PORT,
    }
    if Config.DB_PASSWORD:
        kwargs['password'] = Config.DB_PASSWORD
    
    return psycopg2.connect(**kwargs)

# Verify database and extensions
try:
    conn = get_db_connection()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute('CREATE EXTENSION IF NOT EXISTS postgis')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS extent_cache (
            table_name text PRIMARY KEY,
            minx double precision,
            miny double precision,
            maxx double precision,
            maxy double precision,
            updated_at timestamp
        )
    ''')
    cur.close()
    conn.close()
    print('✓ PostgreSQL/PostGIS verified')
except Exception as e:
    print(f'❌ Database connection failed: {e}')
    raise

In [ ]:
# 6. Cell Size Calculation & Grid Alignment
def compute_cell_size_deg(avg_lat, cell_m=100.0):
    """Compute cell size in degrees for given latitude."""
    cos_lat = max(1e-6, math.cos(math.radians(avg_lat)))
    lon_deg = cell_m / (111320.0 * cos_lat)
    lat_deg = cell_m / 110540.0
    return lon_deg, lat_deg

def align_extent(xmin, ymin, xmax, ymax, origin, cell_lon, cell_lat):
    """Snap extent to global grid."""
    ox, oy = origin
    axmin = ox + math.floor((xmin - ox) / cell_lon) * cell_lon
    aymin = oy + math.floor((ymin - oy) / cell_lat) * cell_lat
    axmax = ox + math.ceil((xmax - ox) / cell_lon) * cell_lon
    aymax = oy + math.ceil((ymax - oy) / cell_lat) * cell_lat
    return axmin, aymin, axmax, aymax

def detect_utm_epsg(xmin, ymin, xmax, ymax):
    """Determine UTM zone EPSG from extent center."""
    clon = (xmin + xmax) / 2.0
    clat = (ymin + ymax) / 2.0
    zone = int(math.floor((clon + 180.0) / 6.0) + 1)
    zone = max(1, min(60, zone))
    return 32600 + zone if clat >= 0 else 32700 + zone

print('✓ Grid alignment functions ready')

In [ ]:
# 7. Main Processing Pipeline
def process_source(base_name, gpkg_path, output_tif):
    """
    Process one source GPKG into a 100m raster.
    
    Pipeline:
    1. Import GPKG to PostGIS (src_table)
    2. Extract polygons, fix invalid geometries
    3. Transform to UTM, compute extent
    4. Generate 100m UTM grid
    5. Compute exact building area per grid cell (clamped to 10,000 m²)
    6. Rasterize with gdal_rasterize
    """
    t0 = time.time()
    
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        src_table = f'src_{base_name}'
        grid_table = f'grid_{base_name}'
        area_table = f'area_{base_name}'
        
        # Import GPKG
        cmd = (
            f'ogr2ogr -f PostgreSQL '
            f'"PG:dbname={Config.DB_NAME} user={Config.DB_USER} host={Config.DB_HOST} port={Config.DB_PORT}" '
            f'--config PG_USE_COPY YES '
            f'"{gpkg_path}" '
            f'-nln {src_table} '
            f'-nlt PROMOTE_TO_MULTI'
        )
        rc, _, err = run_cmd(cmd)
        if rc != 0:
            return False, f'Import failed: {err[-200:]}'
        
        cur.execute(f'CREATE INDEX IF NOT EXISTS {src_table}_geom_idx ON {src_table} USING GIST(geom)')
        conn.commit()
        
        # Get extent and UTM zone
        cur.execute(
            f'SELECT ST_XMin(b), ST_YMin(b), ST_XMax(b), ST_YMax(b) '
            f'FROM (SELECT ST_Extent(ST_SetSRID(geom,4326)) b FROM {src_table}) t'
        )
        row = cur.fetchone()
        if not row or row[0] is None:
            return False, 'No valid geometries'
        
        xmin, ymin, xmax, ymax = map(float, row)
        epsg = detect_utm_epsg(xmin, ymin, xmax, ymax)
        
        # Align raster extent
        avg_lat = (ymin + ymax) / 2.0
        cell_lon, cell_lat = compute_cell_size_deg(avg_lat, Config.CELL_SIZE_M)
        axmin, aymin, axmax, aymax = align_extent(xmin, ymin, xmax, ymax, Config.ALIGN_ORIGIN, cell_lon, cell_lat)
        
        # Extract polygons
        cur.execute(f'DROP TABLE IF EXISTS {grid_table} CASCADE')
        cur.execute(
            f'''
            CREATE TABLE {grid_table} AS
            WITH src AS (
                SELECT CASE WHEN ST_SRID(geom)=0 THEN ST_SetSRID(geom,4326) ELSE geom END AS g
                FROM {src_table}
                WHERE geom IS NOT NULL
            ),
            fixed AS (
                SELECT CASE WHEN NOT ST_IsValid(g) THEN ST_MakeValid(g) ELSE g END AS g2
                FROM src
            ),
            poly AS (
                SELECT ST_CollectionExtract(g2, 3) AS p FROM fixed
            ),
            clean AS (
                SELECT p AS g3 FROM poly WHERE p IS NOT NULL AND NOT ST_IsEmpty(p)
            ),
            utm_box AS (
                SELECT ST_Transform(g3, {epsg}) AS geom FROM clean
            )
            SELECT row_number() OVER() AS gid,
                   ST_MakeEnvelope(
                       floor(ST_XMin(geom)/100)*100,
                       floor(ST_YMin(geom)/100)*100,
                       ceil(ST_XMax(geom)/100)*100,
                       ceil(ST_YMax(geom)/100)*100,
                       {epsg}
                   ) AS extent
            FROM (SELECT ST_Extent(geom) AS geom FROM utm_box) t
            '''
        )
        
        # Generate 100m grid from extent
        cur.execute(
            f'''
            WITH bounds AS (
                SELECT ST_XMin(extent) xmin, ST_YMin(extent) ymin,
                       ST_XMax(extent) xmax, ST_YMax(extent) ymax
                FROM {grid_table}
            )
            DELETE FROM {grid_table};
            
            INSERT INTO {grid_table} (gid, extent)
            SELECT row_number() OVER() AS gid,
                   ST_MakeEnvelope(x, y, x+100, y+100, {epsg}) AS extent
            FROM bounds,
                 generate_series((SELECT xmin FROM bounds)::int, (SELECT xmax FROM bounds)::int, 100) AS x,
                 generate_series((SELECT ymin FROM bounds)::int, (SELECT ymax FROM bounds)::int, 100) AS y
            '''
        )
        
        cur.execute(f'CREATE INDEX ON {grid_table} USING GIST(extent)')
        conn.commit()
        
        # Compute exact area per cell
        cur.execute(f'DROP TABLE IF EXISTS {area_table} CASCADE')
        cur.execute(
            f'''
            CREATE TABLE {area_table} AS
            SELECT g.gid,
                   ST_Transform(g.extent, 4326) AS geom,
                   LEAST(
                       COALESCE(SUM(ST_Area(ST_Intersection(z.geom, g.extent))), 0),
                       {Config.AREA_CLAMP}
                   ) AS building_area_m2
            FROM {grid_table} g
            LEFT JOIN (
                SELECT ST_Transform(ST_SetSRID(geom,4326), {epsg}) AS geom
                FROM {src_table}
            ) z ON ST_Intersects(z.geom, g.extent)
            GROUP BY g.gid, g.extent
            '''
        )
        
        cur.execute(f'CREATE INDEX ON {area_table} USING GIST(geom)')
        conn.commit()
        
        # Rasterize
        ds = f'PG:dbname={Config.DB_NAME} user={Config.DB_USER} host={Config.DB_HOST} port={Config.DB_PORT}'
        cmd = (
            f'gdal_rasterize '
            f'-a building_area_m2 '
            f'-ot Float32 '
            f'-a_nodata -9999 '
            f'-a_srs EPSG:4326 '
            f'-te {axmin} {aymin} {axmax} {aymax} '
            f'-tr {cell_lon} {-cell_lat} '
            f'-co TILED=YES -co COMPRESS=LZW '
            f'-l {area_table} '
            f'"{ds}" '
            f'"{output_tif}"'
        )
        
        rc, _, err = run_cmd(cmd)
        if rc != 0 or not Path(output_tif).exists():
            return False, f'Rasterization failed'
        
        # Cleanup
        for t in [src_table, grid_table, area_table]:
            cur.execute(f'DROP TABLE IF EXISTS {t} CASCADE')
        conn.commit()
        cur.close()
        conn.close()
        
        dt = time.time() - t0
        return True, f'Processed in {dt:.1f}s'
    
    except Exception as e:
        return False, str(e)

print('✓ Processing pipeline ready')

In [ ]:
# 8. Parallel Rasterization
gpkg_files = sorted(Config.GPKG_DIR.glob('*.gpkg'))
print(f'Processing {len(gpkg_files)} GPKG files...')

results = {}
tif_dir = Config.RASTER_DIR / 'tifs'
tif_dir.mkdir(parents=True, exist_ok=True)

with ThreadPoolExecutor(max_workers=Config.WORKERS) as ex:
    futures = {}
    for gpkg in gpkg_files:
        base = safe_name(gpkg.stem)
        tif = tif_dir / f'{base}_100m.tif'
        
        # Skip if already done
        if tif.exists() and tif.stat().st_size > 0:
            results[base] = (True, 'Skipped (already exists)')
            continue
        
        future = ex.submit(process_source, base, gpkg, tif)
        futures[future] = base
    
    for future in as_completed(futures):
        base = futures[future]
        try:
            ok, msg = future.result()
            results[base] = (ok, msg)
            status = '✓' if ok else '✗'
            print(f'{status} {base}: {msg}')
        except Exception as e:
            results[base] = (False, str(e))
            print(f'✗ {base}: {e}')

success_count = sum(1 for ok, _ in results.values() if ok)
print(f'\nProcessing complete: {success_count}/{len(results)} succeeded')

In [ ]:
# 9. Build Mosaic
tifs = sorted(tif_dir.glob('*_100m.tif'))
print(f'Building mosaic from {len(tifs)} rasters...')

vrt_path = tif_dir / 'mosaic.vrt'
mosaic_path = Config.RASTER_DIR / f'mosaic_100m_{RUN_ID}.tif'

if tifs:
    # Build VRT
    cmd = f'gdalbuildvrt -overwrite -resolution highest {vrt_path} ' + ' '.join(str(t) for t in tifs)
    rc, _, err = run_cmd(cmd)
    
    if rc == 0:
        # Convert to GeoTIFF
        cmd = (
            f'gdal_translate -co TILED=YES -co COMPRESS=LZW -co BIGTIFF=YES '
            f'{vrt_path} {mosaic_path}'
        )
        rc, _, err = run_cmd(cmd)
        
        if rc == 0 and mosaic_path.exists():
            size_gb = mosaic_path.stat().st_size / (1024**3)
            print(f'✓ Mosaic created: {mosaic_path.name} ({size_gb:.2f} GB)')
        else:
            print(f'✗ Mosaic creation failed')
    else:
        print(f'✗ VRT creation failed: {err[-200:]}')
else:
    print('No rasters to mosaic')

In [ ]:
# 10. Summary Report
import json
from datetime import datetime

# Calculate statistics
success_count = sum(1 for ok, _ in results.values() if ok)
failed_count = len(results) - success_count

# Analyze raster outputs
raster_stats = {
    'count': 0,
    'total_size_gb': 0.0,
    'extent': {'minx': 180, 'miny': 90, 'maxx': -180, 'maxy': -90}
}

try:
    import rasterio as rio
    for tif in tif_dir.glob('*_100m.tif'):
        with rio.open(tif) as src:
            raster_stats['count'] += 1
            raster_stats['total_size_gb'] += tif.stat().st_size / (1024**3)
            bounds = src.bounds
            raster_stats['extent']['minx'] = min(raster_stats['extent']['minx'], bounds.left)
            raster_stats['extent']['miny'] = min(raster_stats['extent']['miny'], bounds.bottom)
            raster_stats['extent']['maxx'] = max(raster_stats['extent']['maxx'], bounds.right)
            raster_stats['extent']['maxy'] = max(raster_stats['extent']['maxy'], bounds.top)
except Exception as e:
    print(f'(Raster analysis skipped: {e})')

# Create summary JSON
summary = {
    'run_id': RUN_ID,
    'timestamp': datetime.now().isoformat(),
    'sources': {
        'total': len(results),
        'successful': success_count,
        'failed': failed_count,
        'details': {base: {'ok': ok, 'message': msg} for base, (ok, msg) in results.items()}
    },
    'outputs': {
        'rasters': {
            'directory': str(tif_dir),
            'count': raster_stats['count'],
            'total_size_gb': round(raster_stats['total_size_gb'], 2),
            'extent': raster_stats['extent'],
            'projection': 'EPSG:4326',
            'pixel_size': '100m × 100m',
            'data_type': 'Float32',
            'value_range': '0-10000 m²',
            'nodata_value': -9999
        },
        'mosaic': {
            'path': str(mosaic_path) if 'mosaic_path' in locals() else 'Not created',
            'exists': Path(mosaic_path).exists() if 'mosaic_path' in locals() else False,
            'compression': 'LZW',
            'tiling': 'Yes'
        },
        'manifest': str(Config.LOG_DIR / 'raster_manifest.json'),
        'logs': str(Config.LOG_DIR / f'bulk_pipeline_{RUN_ID}.log')
    },
    'configuration': {
        'chunk_size': Config.CHUNK_ROWS,
        'workers': Config.WORKERS,
        'cell_size': '100 meters',
        'grid_origin': Config.ALIGN_ORIGIN,
        'area_clamp': f'{Config.AREA_CLAMP} m²'
    }
}

# Save summary
summary_path = Config.LOG_DIR / f'summary_{RUN_ID}.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'''
╔════════════════════════════════════════════════════════════════╗
║               PIPELINE EXECUTION COMPLETE                      ║
╚════════════════════════════════════════════════════════════════╝

RUN ID: {RUN_ID}

SOURCES PROCESSED:
  Total:       {len(results)}
  Successful:  {success_count}
  Failed:      {failed_count}

RASTER OUTPUTS:
  Location:    {tif_dir}
  Count:       {raster_stats['count']}
  Total size:  {raster_stats['total_size_gb']:.2f} GB
  Format:      GeoTIFF, Float32
  Projection:  EPSG:4326 (WGS84)
  Resolution:  100m × 100m cells
  Coverage:    {raster_stats['extent']}

MOSAIC OUTPUT:
  Path:        {mosaic_path if 'mosaic_path' in locals() else '(not created)'}
  Status:      {'✓ Created' if (mosaic_path.exists() if 'mosaic_path' in locals() else False) else '✗ Failed/Skipped'}

DOCUMENTATION:
  Summary:     {summary_path}
  Pipeline log: {Config.LOG_DIR / f'bulk_pipeline_{RUN_ID}.log'}
  Manifest:    {Config.LOG_DIR / 'raster_manifest.json'}

DATA DETAILS:
  Pixel values:    Building footprint area in m² per 100m cell
  Value range:     0-10,000 m² (clamped)
  No data:         -9999 (cells with no buildings)
  Compression:     LZW
  Grid alignment:  Global origin at (-180°, -90°)

═══════════════════════════════════════════════════════════════════
''')